In [1]:
! pip install unsloth trl datasets transformers bitsandbytes accelerate vllm
! pip install -U transformers trl huggingface_hub mergekit
! pip install flashinfer-cubin==0.6.4

  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.9 MB/s eta 0:00:00
  Using cached flashinfer_cubin-0.6.6-py3-none-any.whl.metadata (1.3 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
Using cached flashinfer_cubin-0.6.6-py3-none-any.whl (267.7 MB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 32.0 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installati

# GRPO CODE

#### Load Model

In [2]:
# 1. UNSLOTH MUST BE IMPORTED FIRST
from unsloth import FastLanguageModel

# 2. Then import everything else
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

max_seq_length = 2048

# Use a BASE model instead of an Instruct model
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/Qwen2.5-0.5B",
    model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False, # <--- SET THIS TO FALSE
)

# Ensure pad token is set for base models (often required for batched generation)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 5.5.3. vLLM: 0.19.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/Qwen3-4B-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


#### Prepare Dataset

In [3]:

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
import re

def format_dataset(row):
    # RELIABLE EXTRACTION: Parse directly from the JSON column, not the prompt string.
    # try:
    examples = row['examples']
    input_tests = [str(ex['input']) for ex in examples]
    output_tests = [str(ex['output']) for ex in examples]
    # except Exception:
    #     input_tests = []
    #     output_tests = []

    # BASE MODEL PROMPT: Force the model into the right format.
    raw_prompt = (
        "You compulsorily have to think and put your reasoning inside <think> </think> tags, and your code inside a markdown block.\n\n"
        f"{row['prompt']}\n\n"
        "<think>\n"
    )

    return {
        "prompt": raw_prompt,
        "input_tests": input_tests,
        "output_tests": output_tests,
    }

print("Loading and filtering dataset...")
# Load dataset and filter for 'A' questions
dataset = load_dataset("open-r1/codeforces-cots", split="train[:1000]")

def filter_difficulty(x):
    return any(x['id'].endswith(lvl) for lvl in ['A', 'B', 'C', 'D'])
filtered_dataset = dataset.filter(filter_difficulty)
processed_dataset = filtered_dataset.map(format_dataset, num_proc=4)

train_dataset = processed_dataset.select(range(min(1000, len(processed_dataset))))


Loading and filtering dataset...


In [4]:
from pprint import pprint
pprint(train_dataset[0]['prompt'])
pprint(train_dataset[0]['input_tests'])
pprint(train_dataset[0]['output_tests'])

('You compulsorily have to think and put your reasoning inside <think> '
 '</think> tags, and your code inside a markdown block.\n'
 '\n'
 'You will be given a competitive programming problem. Please reason step by '
 'step about the solution, then provide a complete implementation in C++17.\n'
 '\n'
 'Your solution must read input from standard input (cin), write output to '
 'standard output (cout).\n'
 'Do not include any debug prints or additional output.\n'
 '\n'
 'Put your final solution within a single code block:\n'
 '```cpp\n'
 '<your code here>\n'
 '```\n'
 '\n'
 '# Problem\n'
 '\n'
 'You are given an array $$$a$$$ of $$$n$$$ integers, where $$$n$$$ is odd.\n'
 '\n'
 'In one operation, you will remove two adjacent elements from the array '
 '$$$a$$$, and then concatenate the remaining parts of the array. For example, '
 'given the array $$$[4,7,4,2,9]$$$, we can obtain the arrays $$$[4,2,9]$$$ '
 'and $$$[4,7,9]$$$ by the operations $$$[\\underline{4,7}, 4,2,9] \\to '
 '[4,2,

#### Define Rewards

In [5]:
# ==========================================
# 2. REWARD FUNCTIONS
# ==========================================
def format_reward_func(completions, **kwargs):
    rewards = []
    for c in completions:
        content = c[0]["content"] if isinstance(c, list) else c

        # BASE MODEL FIX: The prompt already ends with "<think>\n", so the completion
        # will ONLY contain the closing tag. Checking for the opening tag will always fail.
        has_think_close = bool(re.search(r"</think>", content))
        has_code = bool(re.search(r"```(python|cpp)\n.*?```", content, re.DOTALL))

        # Reward shaping: give partial credit if thinking tag is found, full if code exists
        if has_think_close and has_code:
            rewards.append(0.2)
        elif has_think_close or has_code:
            rewards.append(0.1)
        else:
            rewards.append(0.0)
    return rewards

def extract_code(completion):
    match = re.search(r"```(python|cpp)\n(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

def correctness_reward_func(completions, input_tests, output_tests, **kwargs):
    import os, sys, tempfile, subprocess
    rewards = []
    # Zip now correctly pairs each completion with its list of inputs and list of outputs
    for comp, ins, outs in zip(completions, input_tests, output_tests):
        content = comp[0]["content"] if isinstance(comp, list) else comp
        lang, code = extract_code(content)

        # Fail early if there is no code, or if the dataset parser failed to find test cases
        if not code or not ins or len(ins) == 0:
            rewards.append(-0.5)
            continue

        score = 0
        total = len(ins)

        with tempfile.TemporaryDirectory() as tmp:
            if lang == "cpp":
                cpp = os.path.join(tmp, "sol.cpp")
                exe = os.path.join(tmp, "sol")
                with open(cpp, "w") as f: f.write(code)
                
                # COMPILE EXACTLY ONCE
                if subprocess.run(["g++", "-O2", cpp, "-o", exe], capture_output=True).returncode != 0:
                    rewards.append(-0.5) # Compilation Error
                    continue
                cmd = [exe]
            else:
                py = os.path.join(tmp, "sol.py")
                with open(py, "w") as f: f.write(code)
                cmd = [sys.executable, py]

            # EXECUTE ONCE PER TEST CASE
            for i, o in zip(ins, outs):
                try:
                    res = subprocess.run(cmd, input=i, text=True, capture_output=True, timeout=2.0)
                    if res.returncode == 0 and res.stdout.strip() == o.strip():
                        score += 1
                except subprocess.TimeoutExpired:
                    pass
                except Exception:
                    pass

        print(f"Passed {score}/{total} test cases.")
        rewards.append(score / total)

    return rewards

In [6]:
temp_prompt = train_dataset[0]['prompt']

# 2. Tokenize the raw string directly (no chat template needed here)
inputs = tokenizer(
    temp_prompt,
    return_tensors="pt"
).to("cuda")

# 3. Generate the response
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    pad_token_id=tokenizer.eos_token_id,
    use_cache=True
)

# 4. Decode ONLY the newly generated tokens (slice off the input prompt length)
generated_text = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(generated_text)

Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

</think>

</think>

To solve this problem, we need to determine the maximum possible value that can remain in the array after repeatedly removing two adjacent elements until only one element is left.

### Key Observations:
- The array starts with an odd number of elements.
- In each operation, we remove two adjacent elements, and the array size decreases by 2.
- This continues until only one element is left.
- We want to find the **maximum possible value** of the final remaining element.

### Strategy:
This is a **dynamic programming** problem where we need to consider all possible ways to remove pairs of adjacent elements and track the maximum value that can be left.

We can use a **recursive approach with memoization** to explore all possible removals. However, since the array size can be up to 99, and the number of test cases is up to 1000, a recursive approach may be too slow. Therefore, we use **memoization** with a DP table.

### Approach:
1. Use a 2D DP table `dp[i][j]` where `d

In [7]:
! ls

huggingface_tokenizers_cache  outputs  sample_data  unsloth_compiled_cache


In [8]:
# 1. Reconstruct the full text so the regex format checker works properly
full_completion = "<think>\n" + generated_text


# 2. Extract the test cases for this specific problem (index 0)
# Note: The reward functions expect a nested list (a batch containing one list of tests)
input_tests = [processed_dataset[0]["input_tests"]]
output_tests = [processed_dataset[0]["output_tests"]]

print("\n--- Running Evaluation ---")
# 3. Run the Format Reward
fmt_score = format_reward_func([full_completion])[0]
print(f"Format Score      : {fmt_score}")

# 4. Run the Correctness Reward (Compiles and runs the code against the tests)
corr_score = correctness_reward_func([full_completion], input_tests, output_tests)[0]
print(f"Correctness Score : {corr_score}")

# Optional: Print the extracted code to see what was actually compiled
lang, extracted_code = extract_code(full_completion)
if extracted_code:
    print(f"\n--- Extracted {lang.upper()} Code ---")
    print(extracted_code)
else:
    print("\n--- Failed to extract valid code block ---")


--- Running Evaluation ---
Format Score      : 0.2
Passed 1/1 test cases.
Correctness Score : 1.0

--- Extracted CPP Code ---
#include <iostream>
#include <vector>
#include <climits>

using namespace std;

vector<vector<int>> dp;

int maxRemainingValue(vector<int>& a, int i, int j) {
    if (i == j) return a[i];
    if (dp[i][j] != -1) return dp[i][j];
    
    int maxVal = INT_MIN;
    for (int k = i; k < j; k++) {
        int val = max(maxRemainingValue(a, i, k-1), maxRemainingValue(a, k+1, j));
        maxVal = max(maxVal, val);
    }
    dp[i][j] = maxVal;
    return maxVal;
}

int main() {
    int t;
    cin >> t;
    for (int caseNum = 0; caseNum < t; caseNum++) {
        int n;
        cin >> n;
        vector<int> a(n);
        for (int i = 0; i < n; i++) {
            cin >> a[i];
        }
        
        // Initialize DP table
        dp.clear();
        dp.resize(n, vector<int>(n, -1));
        
        // Compute the result
        int result = maxRemainingValue(a, 0, n-

#### Training Loop

In [ ]:

# ==========================================
# 4. TRAINING & LOGGING CALLBACK
# ==========================================
# class EpochSaveCallback(TrainerCallback):
#     def on_epoch_end(self, args, state, control, **kwargs):
#         save_path = os.path.join(args.output_dir, "latest_epoch_weights")
#         print(f"\n[Epoch {state.epoch}] Saving overwritten checkpoint to {save_path}")
#         kwargs["model"].save_pretrained(save_path)
#         kwargs["tokenizer"].save_pretrained(save_path)


training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    num_generations = 4,

    # max_prompt_length = 1024,
    max_completion_length = 2048,
    num_train_epochs = 1,
    save_strategy="steps",
    max_grad_norm = 0.1,
    report_to = "none",
    output_dir = "outputs",
    importance_sampling_level = "sequence",
    loss_type = "dr_grpo",
    # save_strategy="steps",   # or "epoch"
    save_steps=2,          # save every 1 steps
    save_total_limit=2,      # keep last 3 checkpoints
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
    # callbacks=[EpochSaveCallback()]
)

print("Starting GRPO Training...")
trainer.train()

# Add this to manually save the final result
trainer.save_model("outputs/final_grpo_model")
tokenizer.save_pretrained("outputs/final_grpo_model")


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting GRPO Training...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 235 | Num Epochs = 1 | Total steps = 58
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)
Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence.

Passed 0/1 test cases.
Passed 1/1 test cases.
Passed 1/1 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 1/3 test cases.
Passed 1/3 test cases.
Passed 0/3 test cases.
Passed 0/3 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward_func / mean,rewards / format_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,-0.021926,0.366667,0.344265,722.125000,511.000000,969.000000,0.000000,722.125000,511.000000,969.000000,0.000000,0.200000,0.000000,0.166667,0.344265
2,0.013277,0.752083,0.466344,415.062500,250.000000,646.000000,0.000000,415.062500,250.000000,646.000000,0.000000,0.200000,0.000000,0.552083,0.466344
3,-0.002933,0.502083,0.381123,496.812500,342.000000,668.000000,0.000000,496.812500,342.000000,668.000000,0.000007,0.200000,0.000000,0.302083,0.381123


...

Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Passed 0/3 test cases.
Passed 3/3 test cases.
Passed 3/3 test cases.
Passed 0/3 test cases.
Passed 2/3 test cases.
Passed 0/3 test cases.
Passed 2/3 test cases.
Passed 0/3 test cases.
Passed 0/3 test cases.
Passed 0/3 test cases.
Passed 0/3 test cases.
Passed 0/3 test cases.
Passed 1/2 test cases.
Passed 0/2 test cases.
Passed 1/2 test cases.
Passed 1/2 test cases.


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 0/1 test cases.
Passed 1/3 test cases.
Passed 1/3 test cases.
Passed 1/3 test cases.
Passed 3/4 test cases.
Passed 4/4 test cases.
Passed 2/4 test cases.
Passed 3/4 test cases.
Passed 5/6 test cases.
Passed 3/6 test cases.
Passed 3/6 test cases.
Passed 3/6 test cases.


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

#### Post Training Inference

In [ ]:

# ==========================================
# 5. POST-TRAINING VERIFICATION
# ==========================================
print("\n--- Running Manual Verification ---")
val_sample = processed_dataset[-1]

# BASE MODEL CHANGE: Use direct text tokenization instead of apply_chat_template
prompt_text = val_sample["prompt"]
inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

print(f"Generating solution for Problem: {val_sample['id']}")
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    pad_token_id=tokenizer.pad_token_id
)

# Decode only the generated portion
generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("\n--- Generated Output ---")
print(generated_text)
print("------------------------")

format_score = format_reward_func([generated_text])[0]
correctness_score = correctness_reward_func(
    [generated_text],
    [val_sample["input_tests"]],
    [val_sample["output_tests"]]
)[0]

print("\n--- Metrics ---")
print(f"Format Reward  : {format_score}")
print(f"Correctness    : {correctness_score} (Percentage of Test Cases Passed)")